# Sweep: MEAN Pooling - Negation Detection

Train probes to detect **negation** (NOT sentiment!) across all layers using MEAN pooling.

**Task**: Binary classification
- Label 0: No negation (e.g., "good", "the movie is great")
- Label 1: Has negation (e.g., "not good", "the movie is not great")

**Goal**: Find which layer is best at detecting negation. Expected: Layer 3 (based on cosine similarity findings).

Run this notebook in **Colab 2** while running `07_sweep_cls_negation_detection.ipynb` and `07_sweep_token_negation_detection.ipynb` in separate Colab runtimes.

All results are saved to Google Drive so they can be merged later.


## 1. Setup


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone or pull the repository
!git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
%cd Negation-Origin-Tracing


In [ ]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm


In [ ]:
# Set shared output location on Google Drive
import os
os.environ['DRIVE_OUTPUT'] = '/content/drive/MyDrive/NOT_results'

# Create the directory
!mkdir -p /content/drive/MyDrive/NOT_results

print(f"Results will be saved to: {os.environ['DRIVE_OUTPUT']}")


In [ ]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Prepare Negation Detection Dataset

Download JinaAI negation dataset and convert to detection format:
- Anchor/Entailment sentences → Label 0 (no negation)
- Negative sentences → Label 1 (has negation)


In [ ]:
# Check if negation detection dataset already exists
import os

train_path = 'data/raw/train/negation_detection.parquet'
val_path = 'data/raw/validation/negation_detection.parquet'
test_path = 'data/raw/test/negation_detection.parquet'

if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
    print("Negation detection dataset already exists!")
    
    # Show stats
    import pandas as pd
    for name, path in [('Train', train_path), ('Validation', val_path), ('Test', test_path)]:
        df = pd.read_parquet(path)
        print(f"  {name}: {len(df)} samples, labels: {sorted(df['label'].unique())}")
else:
    print("Preparing negation detection dataset...")
    !python src/data/prepare_negation_detection.py


## 3. Run MEAN Pooling Sweep - Negation Detection

This trains probes on all 6 layers using MEAN pooling to detect negation.


In [ ]:
# Run the MEAN negation detection sweep
!chmod +x run_sweep_mean_negation_detection.sh
!./run_sweep_mean_negation_detection.sh


## 4. Check Results


In [ ]:
import json
import os

results_file = os.path.join(os.environ['DRIVE_OUTPUT'], 'sweep_mean_negation_detection', 'results_mean_negation_detection.json')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print(f"MEAN Negation Detection Sweep Results ({len(results)} experiments)")
    print("=" * 60)
    print("\nTask: Negation Detection (0=no negation, 1=has negation)")
    print("\nResults by layer (sorted by AUROC):")
    
    for r in sorted(results, key=lambda x: x.get('test_auroc', 0), reverse=True):
        print(f"  Layer {r['layer']}: AUROC={r.get('test_auroc', 0):.4f}, Acc={r.get('test_acc', 0):.4f}")
    
    best = max(results, key=lambda x: x.get('test_auroc', 0))
    print(f"\n🏆 Best: Layer {best['layer']} with AUROC {best.get('test_auroc', 0):.4f}")
    
    # Check if Layer 3 is best (expected based on cosine similarity)
    if best['layer'] == 3:
        print("\n✅ Layer 3 is best for negation detection!")
        print("   This connects with the cosine similarity findings.")
    else:
        print(f"\n⚠️ Layer {best['layer']} is best (not Layer 3)")
        print("   This may suggest a different pattern than cosine similarity.")
else:
    print(f"Results not found at {results_file}")


In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import json
import os

results_file = os.path.join(os.environ['DRIVE_OUTPUT'], 'sweep_mean_negation_detection', 'results_mean_negation_detection.json')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    # Sort by layer
    results = sorted(results, key=lambda x: x['layer'])
    
    layers = [r['layer'] for r in results]
    aurocs = [r.get('test_auroc', 0) for r in results]
    accs = [r.get('test_acc', 0) for r in results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # AUROC plot
    bars1 = ax1.bar(layers, aurocs, color='steelblue', alpha=0.8)
    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Test AUROC')
    ax1.set_title('Negation Detection: MEAN Pooling - AUROC by Layer')
    ax1.set_xticks(layers)
    ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
    ax1.legend()
    
    # Highlight best layer
    best_idx = aurocs.index(max(aurocs))
    bars1[best_idx].set_color('green')
    
    # Accuracy plot
    bars2 = ax2.bar(layers, accs, color='coral', alpha=0.8)
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Test Accuracy')
    ax2.set_title('Negation Detection: MEAN Pooling - Accuracy by Layer')
    ax2.set_xticks(layers)
    ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
    ax2.legend()
    
    # Highlight best layer
    best_idx = accs.index(max(accs))
    bars2[best_idx].set_color('green')
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(os.environ['DRIVE_OUTPUT'], 'sweep_mean_negation_detection', 'mean_negation_detection_results.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Figure saved to: {fig_path}")
    
    plt.show()
else:
    print("No results to visualize")
